In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/gwo-pso-knn")
else:
    RESULT_DIR = Path.cwd() / "results" / "gwo-pso-knn"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
import time
import json
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from typing import Tuple, List, Dict, Optional

In [2]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [3]:
X_path = resolve_data_path("X_train.csv")
y_path = resolve_data_path("y_train.csv")

X, y = load_data(X_path, y_path)

In [4]:
X_data = X.values if hasattr(X, "values") else X
y_data = y.values.ravel() if hasattr(y, "values") else np.ravel(y)

X_tr_all, X_val_all, y_tr, y_val = train_test_split(
    X_data,
    y_data,
    test_size=0.2,
    random_state=42,
    stratify=y_data
)

In [5]:


def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def fitness(solution, alpha=0.9, beta=0.1):
    if np.sum(solution) == 0:
        return 1  

    # chọn feature theo mask
    X_tr = X_tr_all[:, solution == 1]
    X_val = X_val_all[:, solution == 1]

    if X_tr.shape[1] == 0:
        return 1

    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_tr, y_tr)

    accuracy = model.score(X_val, y_val)

    feature_ratio = np.sum(solution) / len(solution)

    return alpha * (1 - accuracy) + beta * feature_ratio

In [6]:
def hybrid_gwo_pso_with_population(n_agents=20, max_iter=100):
    n_features = X.shape[1]
    positions = np.random.rand(n_agents, n_features)
    velocities = np.zeros_like(positions)
    a_start, a_end = 2, 0  

    alpha_pos = beta_pos = delta_pos = np.zeros(n_features)
    alpha_score = beta_score = delta_score = float('inf')

    history = {
        "list_global_best_fit": [],
        "list_current_best_fit": [],
        "list_diversity": [],
        "list_exploration": [],
        "list_exploitation": [],
        "list_epoch_time": [],
        "list_population": []
    }

    for iteration in range(max_iter):
        start_time = time.time()
        previous_masks_for_metrics = continuous_population_to_masks(
            positions,
            threshold=0.5,
            min_features=1,
            use_sigmoid=True
        )
        current_best_score = float('inf')
        scores = []
        for i in range(n_agents):
            binary_solution = (sigmoid(positions[i]) >= 0.5).astype(int)
            score = fitness(binary_solution)
            scores.append(score)
            if score < alpha_score:
                alpha_score, alpha_pos = score, positions[i].copy()
            elif score < beta_score:
                beta_score, beta_pos = score, positions[i].copy()
            elif score < delta_score:
                delta_score, delta_pos = score, positions[i].copy()
            if score < current_best_score:
                current_best_score = score

        a = a_start - (a_start - a_end) * (iteration / max_iter) ** 2

        for i in range(n_agents):
            r1, r2 = np.random.rand(2)
            A1, C1 = 2 * a * r1 - a, 2 * r2
            D_alpha = np.abs(C1 * alpha_pos - positions[i])
            X1 = alpha_pos - A1 * D_alpha

            r1, r2 = np.random.rand(2)
            A2, C2 = 2 * a * r1 - a, 2 * r2
            D_beta = np.abs(C2 * beta_pos - positions[i])
            X2 = beta_pos - A2 * D_beta

            r1, r2 = np.random.rand(2)
            A3, C3 = 2 * a * r1 - a, 2 * r2
            D_delta = np.abs(C3 * delta_pos - positions[i])
            X3 = delta_pos - A3 * D_delta

            positions[i] = (X1 + X2 + X3) / 3  
            velocities[i] += np.random.rand() * (alpha_pos - positions[i])  
            positions[i] += velocities[i]

        # Calculate metrics using the paper formulas on binary feature masks.
        current_masks_for_metrics = continuous_population_to_masks(
            positions,
            threshold=0.5,
            min_features=1,
            use_sigmoid=True
        )
        diversity = compute_population_diversity(current_masks_for_metrics)
        exploration, exploitation = compute_exploration_exploitation(
            new_masks=current_masks_for_metrics,
            previous_masks=previous_masks_for_metrics,
            similarity_threshold=0.1
        )

        history["list_current_best_fit"].append(current_best_score)
        history["list_global_best_fit"].append(alpha_score)
        history["list_diversity"].append(diversity)
        history["list_exploration"].append(exploration)
        history["list_exploitation"].append(exploitation)
        history["list_epoch_time"].append(time.time() - start_time)
        history["list_population"].append(positions.copy().tolist())

        print(f"Iteration {iteration + 1} - Best Fitness: {alpha_score:.4f} | Best Current Fitness: {current_best_score:.5f} | Diversity: {diversity:.4f} | Exploration: {exploration:.4f} | Exploitation: {exploitation:.4f} | Time: {history['list_epoch_time'][-1]:.4f}s")

    df = pd.DataFrame({
        "epoch": list(range(1, max_iter + 1)),
        "global_best_fit": history["list_global_best_fit"],
        "current_best_fit": history["list_current_best_fit"],
        "diversity": history["list_diversity"],
        "exploration": history["list_exploration"],
        "exploitation": history["list_exploitation"],
        "epoch_time": history["list_epoch_time"]
    })

    df.to_csv(f"optimization_metrics_gwopso_with_formulas.csv", index=False)
    print(f"Saved optimization_metrics_gwopso_with_formulas.csv")

    return (sigmoid(alpha_pos) >= 0.5).astype(int)

In [7]:
start_train = time.time()
selected_features = hybrid_gwo_pso_with_population()
end_train = time.time()

print(f"Feature selection time: {end_train - start_train:.4f} seconds")
print("Selected Features Index:", np.where(selected_features == 1)[0])
print("Number of Selected Features:", np.sum(selected_features))

Iteration 1 - Best Fitness: 0.5458 | Best Current Fitness: 0.54578 | Time: 41.6283s
Iteration 2 - Best Fitness: 0.5393 | Best Current Fitness: 0.53928 | Time: 40.8381s
Iteration 3 - Best Fitness: 0.5170 | Best Current Fitness: 0.51699 | Time: 36.5000s
Iteration 4 - Best Fitness: 0.5113 | Best Current Fitness: 0.51129 | Time: 30.4283s
Iteration 5 - Best Fitness: 0.5029 | Best Current Fitness: 0.50294 | Time: 28.8605s
Iteration 6 - Best Fitness: 0.5029 | Best Current Fitness: 0.50415 | Time: 21.3407s
Iteration 7 - Best Fitness: 0.5029 | Best Current Fitness: 0.50661 | Time: 11.9009s
Iteration 8 - Best Fitness: 0.5029 | Best Current Fitness: 0.50872 | Time: 12.2469s
Iteration 9 - Best Fitness: 0.5029 | Best Current Fitness: 0.51158 | Time: 12.8343s
Iteration 10 - Best Fitness: 0.5029 | Best Current Fitness: 0.51058 | Time: 14.1966s
Iteration 11 - Best Fitness: 0.5029 | Best Current Fitness: 0.51324 | Time: 13.9744s
Iteration 12 - Best Fitness: 0.5029 | Best Current Fitness: 0.50853 | Time

In [8]:
import numpy as np

selected_indices = np.where(selected_features == 1)[0]

print(selected_indices)

[  1   2   3   9  10  11  12  13  14  15  22  26  27  30  31  32  33  34
  35  36  41  42  43  46  47  52  54  55  59  61  64  65  69  73  74  76
  77  80  81  83  84  85  86  87  88  89  91  96  98  99 103 104 107 108
 109 111 114 115 117 118 122 123 125 126 128 130 133 134 135 136 137 139
 140 141 144 145 146 147 151 154 155 156 157 158 159 160 162 163 164 165
 169 171 173 175 176 179 180 181 182 183 184 187 188 189 192 194 198 201
 204 207 209 213 218 223 224 227 229 231 233 236 238 239 244 251 255 257
 259 260 262 264 265 275 278 279 280 281 282 285 288 289 291 293 295 297
 300 301 306 310 311 312 314 315 316 317 318 319 320 321 322 323 325 326
 327 328 329 331 332 334 335 336 339 349 350 352 355 363 365 369 372 374
 376 379 380 386 387 388 391 394 395 399 401 402 403 408 410 411 416 418
 419 423 424 426 431 432 435 436 438 444 447 450 451 453 467 470 473 483
 484 486 487 488 492 500 501 502 503 504 507 514 517 518 520 523 525 526
 528 531 533 536 540 547 548 551 553 554 556 558 55

In [9]:
with open(output_path('selected_indices.csv'), 'w') as f:
    for idx in selected_indices:
        f.write(f"{idx}\n")

In [10]:
import pandas as pd

selected_features = pd.read_csv(output_path("selected_indices.csv"), header=None)[0].values

X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

X_train_selected = X_train.iloc[:, selected_features].copy()
X_val_selected = X_val.iloc[:, selected_features].copy()
X_test_selected = X_test.iloc[:, selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 433)
Val selected shape: (2000, 433)
Test selected shape: (2000, 433)


In [11]:
def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [12]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5233

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 10}
  CV F1-macro : 0.5582

[RandomForest] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.6938

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3372

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'saga'}
  CV F1-macro : 0.6976

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 10}
  CV F1-macro : 0.6930

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.7781


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5445,0.523308,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.446809,0.168,0.244186,250.0,0.420732,0.828,...,0.682243,250.0,0.572718,0.5445,0.531137,2000.0,0.572718,0.5445,0.531137,2000.0
1,DecisionTree,0.5615,0.558164,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.346774,0.344,0.345382,250.0,0.630282,0.716,...,0.492632,250.0,0.572691,0.5615,0.562482,2000.0,0.572691,0.5615,0.562482,2000.0
2,RandomForest,0.7090,0.693754,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.611511,0.340,0.437018,250.0,0.701299,0.864,...,0.736607,250.0,0.716286,0.7090,0.701236,2000.0,0.716286,0.7090,0.701236,2000.0
3,GaussianNB,0.3505,0.337202,{'var_smoothing': 1e-06},0.125000,0.036,0.055901,250.0,0.328244,0.860,...,0.478149,250.0,0.371026,0.3505,0.327615,2000.0,0.371026,0.3505,0.327615,2000.0
4,LogisticRegression,0.7095,0.697639,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.597938,0.464,0.522523,250.0,0.740072,0.820,...,0.736220,250.0,0.704952,0.7095,0.705491,2000.0,0.704952,0.7095,0.705491,2000.0
5,LinearSVM,0.7075,0.692972,{'C': 10},0.615854,0.404,0.487923,250.0,0.727891,0.856,...,0.731225,250.0,0.702206,0.7075,0.700713,2000.0,0.702206,0.7075,0.700713,2000.0
6,RBF_SVM,0.7870,0.778098,"{'C': 10, 'gamma': 'auto'}",0.665198,0.604,0.633124,250.0,0.805147,0.876,...,0.839757,250.0,0.786143,0.7870,0.785759,2000.0,0.786143,0.7870,0.785759,2000.0
